In [4]:
#1. Creating data summary --grid cell level annual average exceedance over 30 year block , also attaching NRM region and masking with land surface 
#This step uses decadal summary data produced by 05 supplimentary_variable_analysis_metrics
import xarray as xr
import pandas as pd
import geopandas as gpd
import regionmask

# --------------------------------------------------
# Setup (run once)
# --------------------------------------------------

# NRM regions
nrm_gdf = gpd.read_file(
    "/scratch/dt6/xw9650/tmp/reference_data/NRM_clusters/NRM_clusters.shp"
)
nrm_gdf = nrm_gdf[nrm_gdf.geometry.notnull()].to_crs("EPSG:4326")

nrm_names = nrm_gdf["label"].tolist()

nrm_regions = regionmask.Regions(
    name="NRM",
    numbers=list(range(len(nrm_gdf))),
    names=nrm_names,
    outlines=nrm_gdf.geometry.tolist(),
)

# Land mask
land_mask_path = (
    "/scratch/dt6/xw9650/tmp/regridded/"
    "sftlf_AUS-18_ACCESS-ESM1-5_historical_r6i1p1f1_"
    "NSW-Government_NARCliM2-0-WRF412R3_v1-r1_fx.nc"
)

sftlf = xr.open_dataset(land_mask_path)["sftlf"]

def process_member(
    infile,
    gcm,
    rcm,
    scenario,
    nrm_regions,
    nrm_names,
    sftlf,
    var='wbgt'
):

    ds = xr.open_dataset(infile)

    # NRM mask
    nrm_id = nrm_regions.mask(ds)

    # Land mask
    land_mask = (
        sftlf.interp(
            lat=ds.lat,
            lon=ds.lon,
            method="nearest",
        )
        > 50
    )

    nrm_id_land = nrm_id.where(land_mask)

    da = ds[var].where(land_mask)

    da = da.assign_coords(
        nrm_region=(("lat", "lon"), nrm_id_land.values)
    )

    df = (
        da.to_dataframe(name=var)
        .reset_index()
        .dropna(subset=[var, "nrm_region"])
    )

    df["region"] = (
        df["nrm_region"]
        .astype(int)
        .map(dict(enumerate(nrm_names)))
    )

    # Extract start year
    df["period"] = pd.to_datetime(
        df["period"]
    ).dt.year
    
    # Create block labels
    block_lookup = {
        1981: "1981-2010",
        2036: "2036-2065",
        2071: "2071-2100",
    }

    df["block"] = df["period"].map(block_lookup)
    
    # Optional: remove period column entirely
    df = df.drop(columns="period")

    df["GCM"] = gcm
    df["RCM"] = rcm
    df["scenario"] = scenario

    return df

def process_scenario(
    scenario,
    gcms,
    rcms,
    nrm_regions,
    nrm_names,
    sftlf,
    input_dir="/scratch/dt6/xw9650/tmp/",
    output_dir="/scratch/dt6/xw9650/tmp/model_intercomparison_plots/",
    var='wbgt'
):

    all_dfs = []

    for gcm in gcms:
        for rcm in rcms:

            infile = (
                f"{input_dir}/"
                f"{var}_{gcm}_{rcm}_{scenario}_decadal_summary.nc"
            )

            try:

                df = process_member(
                    infile=infile,
                    gcm=gcm,
                    rcm=rcm,
                    scenario=scenario,
                    nrm_regions=nrm_regions,
                    nrm_names=nrm_names,
                    sftlf=sftlf,
                    var=var,
                )

                all_dfs.append(df)

                print(f"Finished {scenario}: {gcm} {rcm}")

            except FileNotFoundError:

                print(f"Missing {scenario}: {gcm} {rcm}")

    if not all_dfs:
        raise ValueError(
            f"No files found for scenario {scenario}"
        )

    combined_df = pd.concat(
        all_dfs,
        ignore_index=True
    )

    combined_df = (
        combined_df
        .drop(columns=["height", "quantile"], errors="ignore")
        .assign(exceed_days=lambda x: x[var] / 30)
        .drop(columns=var)
        .rename(columns={"period": "block"})
    )

    outfile = (
        f"{output_dir}/"
        f"{var}_{scenario}_all_members.csv"
    )

    combined_df.to_csv(outfile, index=False)

    print(f"Saved {outfile}")

    return combined_df



In [ ]:
#1.1 (used for paper& thesis) 
# execute function wrapper 
gcms = [
    "ACCESS-ESM1-5",
    "EC-Earth3-Veg",
    "MPI-ESM1-2-HR",
    "NorESM2-MM",
    "UKESM1-0-LL",
]

rcms = [
    "NARCliM2-0-WRF412R3",
    "NARCliM2-0-WRF412R5",
]

df_ssp126 = process_scenario(
    "ssp126",
    gcms,
    rcms,
    nrm_regions,
    nrm_names,
    sftlf,
)

df_ssp245 = process_scenario(
    "ssp245",
    gcms,
    rcms,
    nrm_regions,
    nrm_names,
    sftlf,
)

df_ssp370 = process_scenario(
    "ssp370",
    gcms,
    rcms,
    nrm_regions,
    nrm_names,
    sftlf,
)

In [7]:
#2. plotting function: plotting exceedance counts from all model ensambles in one graph --with median and 90 pc trends 
#Stacked inter-model comparison plots showing how exceedance days vary across different regions, climate models (GCMs/RCMs), and time periods (blocks)
#for each variable and scenario.

import matplotlib.pyplot as plt
import seaborn as sns
import os
import math
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator



def plot_inter_model_comparison(
    df_all,
    var,
    scenario,
    gcms,
    rcms,
    blocks,
    output_dir,
    shared_ylabel=None,
    file_suffix=None 
):
    """
    Generate stacked boxplots for inter-model comparison across regions,
    with subplots arranged in 2 columns.
    Adds lines showing changes in median (solid) #removed 90th percentile (dashed)
    from historical → mid-century → end-century, and a combined legend.
    """
    # Determine variable label and shared y-axis label
    filename_var = "Wet&Windy" if var == "joint_exceed" else var
    if shared_ylabel is None:
        if var == "joint_exceed":
            shared_ylabel = "Exceedance Days per Decade (per gridcell)"
        else:
            shared_ylabel = "Exceedance Days per Year (per gridcell)"

    df_all["GCM_block"] = df_all["GCM"] + " | " + df_all["block"]


    regions = list(df_all["region"].unique())
    n_regions = len(regions)

    
    # Arrange subplots in 2 columns
    n_cols = 2
    n_rows = math.ceil(n_regions / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 5*n_rows), sharex=False, sharey=True)#modified 
    axes = axes.flatten()


    # Order x-axis: for each GCM, cycle through blocks
    order = [f"{gcm} | {block_name}" for gcm in gcms for block_name in blocks.keys()]


    global_ymax = np.percentile(df_all["exceed_days"], 99.9)
    global_ymin = 0
    y_buffer = 0.05 * global_ymax   # small padding so lines don’t touch the top
    
    

    # Build color palette for RCMs
    palette = sns.color_palette("Set2", n_colors=len(rcms))
    color_map = dict(zip(rcms, palette))

    for idx, ax in enumerate(axes):
        if idx >= n_regions:
            ax.axis('off')
            continue

        region_name = regions[idx]
        subset = df_all[df_all["region"] == region_name]

        sns.boxplot(
            data=subset,
            x="GCM_block", y="exceed_days", hue="RCM",
            palette=color_map,
            order=order, ax=ax,
            width=0.5,
            linewidth=1.2,
            showfliers=False,       # hide outliers
            whis=(10, 90)  
        )

        ax.set_title(f"{region_name}", fontsize=24, fontweight="bold", pad=25)
        ax.set_ylabel("")
        ax.set_xlabel("")

        # Replace x tick labels with just block names
        ticks = []
        labels = []
        for gcm in gcms:
            for block_name in blocks.keys():
                ticks.append(f"{gcm} | {block_name}")
                labels.append(block_name)
        ax.set_xticks(range(len(ticks)))
        ax.set_xticklabels(labels, rotation=45, ha="center", fontsize=20)

        ax.set_ylim(global_ymin, global_ymax + y_buffer)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=8))
        ax.tick_params(axis='y', labelsize=20)


        # Add GCM labels above each block group — only on top row
        blocks_per_gcm = len(blocks)
        if idx < n_cols:  # top row only
            for i, gcm in enumerate(gcms):
                start = i * blocks_per_gcm
                end = start + blocks_per_gcm - 1
                mid = (start + end) / 2
                ax.text(
                    mid, global_ymax + y_buffer * 1,
                    gcm, ha="center", va="bottom",
                    fontsize=18, fontweight="bold"
                )

        # ---- Compute median + P90 and plot connecting lines ----
        stats_data = (
            subset.groupby(["GCM", "RCM", "block"])["exceed_days"]
            .agg(median="median")#, p90=lambda x: np.percentile(x, 90)) ### 90th percentile trend is removed 
            .reset_index()
        )

        block_names = list(blocks.keys())
        x_positions = {
            f"{gcm} | {b}": i
            for i, (gcm, b) in enumerate([(g, bl) for g in gcms for bl in block_names])
        }

        for (gcm, rcm), sub in stats_data.groupby(["GCM", "RCM"]):
            sub_sorted = sub.set_index("block").reindex(block_names).reset_index()

            for i in range(len(block_names) - 1):
                b1, b2 = block_names[i], block_names[i+1]

                # Median line (solid)
                if not pd.isna(sub_sorted.loc[i, "median"]) and not pd.isna(sub_sorted.loc[i+1, "median"]):
                    x1, x2 = x_positions[f"{gcm} | {b1}"], x_positions[f"{gcm} | {b2}"]
                    y1, y2 = sub_sorted.loc[i, "median"], sub_sorted.loc[i+1, "median"]
                    ax.plot([x1, x2], [y1, y2],
                            color=color_map.get(rcm, "black"),
                            linewidth=1.5, alpha=0.4, zorder=6)


        # Only bottom row subplots get x-axis labels
        if idx < n_cols*(n_rows-1):
            ax.set_xticks([])
            ax.set_xticklabels([])

        # Add vertical separators between GCMs
        for i in range(blocks_per_gcm, len(gcms)*blocks_per_gcm, blocks_per_gcm):
            ax.axvline(i-0.5, color="grey", linestyle="--", alpha=0.6)

        # ---- Combined legend (only once) ----
        if idx == 0:
            # First subplot keeps RCM legend
            rcm_legend = ax.legend(fontsize=14, title_fontsize=16)

            # Add line-type legend handles
            line_handles = [
                Line2D([0], [0], color="black", linewidth=1.5, linestyle="-", label="Median change")#,
                # Line2D([0], [0], color="black", linewidth=1.5, linestyle="--", label="90th percentile change"),  ##removed for 90th percentile
            ]

            # Combine legends (RCM + line-style)
            handles, labels = ax.get_legend_handles_labels()
            combined_handles = handles + line_handles
            combined_labels = labels + ["Median change"]#, "90th percentile change"]

            ax.legend(combined_handles, combined_labels,
                      fontsize=18, title_fontsize=20, loc="upper right")
        else:
            ax.legend_.remove()

    # Shared y-axis label (conditional)
    fig.text(
        0.02, 0.5, shared_ylabel,
        va='center', ha='center',
        rotation='vertical', fontsize=26, fontweight="bold"
    )

    # Shared figure title
    scenario_label = {
        "ssp126": "SSP1-2.6",
        "ssp245": "SSP2-4.5",
        "ssp370": "SSP3-7.0",
    }
    


    plt.tight_layout(rect=[0.05, 0, 0.95, 0.94])
     
    # 🔹 Move subplot titles slightly higher (after tight_layout)
    for ax in axes[:n_regions]:
        ax.title.set_position([0.5, 1.1])
        
    # Save figure
    out_file = os.path.join(output_dir, f"{filename_var}_exceed_{scenario}_allRegions_allMembers.png")
    plt.savefig(out_file, dpi=150)
    plt.close()
    print(f"📊 Saved stacked plot with median + P90 trend lines and combined legend -> {out_file}")


def run_model_intercomparison(variables, scenarios, output_dir):
    for var in variables:
        for scenario in scenarios:
            # build file path
            filename_var = "Wet&Windy" if var == "joint_exceed" else var 
            file_path = os.path.join(
                output_dir, f"{filename_var}_{scenario}_all_members.csv"
            )
            
            if not os.path.exists(file_path):
                print(f"⚠️ File not found for {filename_var} {scenario}: {file_path}")
                continue
            
            # load parquet file
            df = pd.read_csv(file_path)
            
            # call plotting function
            plot_inter_model_comparison(
                df_all=df,
                var=var,
                scenario=scenario,
                gcms=["ACCESS-ESM1-5", "EC-Earth3-Veg","MPI-ESM1-2-HR", "NorESM2-MM", "UKESM1-0-LL"],
                rcms=["NARCliM2-0-WRF412R3", "NARCliM2-0-WRF412R5"],
                blocks={
                    "1981-2010": slice(1981, 2010),
                    "2036-2065": slice(2036, 2065),
                    "2071-2100": slice(2071, 2100),
                },
                output_dir=output_dir
            )



In [2]:
#2.1 execute plot for varibles (used for paper& thesis ) Paper Figure 7, S6, S7 / Thesis Figure 4.6, B.6, B.7

#ssp126
variables = ["wbgt"]
scenarios = [ "ssp126"]

run_model_intercomparison(
    variables=variables,
    scenarios=scenarios,
    output_dir="/scratch/dt6/xw9650/tmp/model_intercomparison_plots/"
)

#ssp245
variables = ["wbgt"]
scenarios = [ "ssp245"]

run_model_intercomparison(
    variables=variables,
    scenarios=scenarios,
    output_dir="/scratch/dt6/xw9650/tmp/model_intercomparison_plots/"
)

#ssp370
variables = ["wbgt"]
scenarios = [ "ssp370"]

run_model_intercomparison(
    variables=variables,
    scenarios=scenarios,
    output_dir="/scratch/dt6/xw9650/tmp/model_intercomparison_plots/"
)


📊 Saved stacked plot with median + P90 trend lines and combined legend -> /scratch/dt6/xw9650/tmp/model_intercomparison_plots/wbgt_exceed_ssp126_allRegions_allMembers.png
📊 Saved stacked plot with median + P90 trend lines and combined legend -> /scratch/dt6/xw9650/tmp/model_intercomparison_plots/wbgt_exceed_ssp245_allRegions_allMembers.png
📊 Saved stacked plot with median + P90 trend lines and combined legend -> /scratch/dt6/xw9650/tmp/model_intercomparison_plots/wbgt_exceed_ssp370_allRegions_allMembers.png


In [9]:
#3. generate table of statistics 
import os
import numpy as np
import pandas as pd


def generate_boxplot_statistics(
    input_csv,
    output_csv
):
    """
    Calculate boxplot statistics used in the inter-model comparison plots.

    Produces:
        region
        GCM
        RCM
        block
        p10
        median
        p90
        mean
        min
        max
        n
    """

    print(f"Reading: {input_csv}")

    df = pd.read_csv(input_csv)

    required_cols = [
        "region",
        "GCM",
        "RCM",
        "block",
        "exceed_days"
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    stats_df = (
        df.groupby(
            ["region", "GCM", "RCM", "block"]
        )["exceed_days"]
        .agg(
            p10=lambda x: np.percentile(x, 10),
            median="median",
            p90=lambda x: np.percentile(x, 90),
            mean="mean",
            min="min",
            max="max",
            n="count"
        )
        .reset_index()
    )
    
    # Round numeric statistics to nearest integer
    cols_to_round = [
        "p10",
        "median",
        "p90",
        "mean",
        "min",
        "max"
    ]
    
    stats_df[cols_to_round] = (
        stats_df[cols_to_round]
        .round(0)
        .astype(int)
    )
    stats_df = stats_df.sort_values(
        ["region", "GCM", "RCM", "block"]
    )

    stats_df.to_csv(output_csv, index=False)

    print(f"Saved statistics table:")
    print(output_csv)

    print("\nFirst few rows:")
    print(stats_df.head())

    return stats_df


In [ ]:
#3.1 (produced for paper) 

###############################################################################
# RUN
###############################################################################

output_dir = r"/scratch/dt6/xw9650/tmp/model_intercomparison_plots/"

variables = [
    "wbgt",

    # add others if needed
]

scenarios = [
    "ssp126","ssp245","ssp370"
]

for var in variables:

    filename_var = (
        "Wet&Windy"
        if var == "joint_exceed"
        else var
    )

    for scenario in scenarios:

        input_csv = os.path.join(
            output_dir,
            f"{filename_var}_{scenario}_all_members.csv"
        )

        if not os.path.exists(input_csv):
            print(f"Skipping missing file: {input_csv}")
            continue

        output_csv = os.path.join(
            output_dir,
            f"{filename_var}_{scenario}_boxplot_statistics.csv"
        )

        generate_boxplot_statistics(
            input_csv=input_csv,
            output_csv=output_csv
        )

print("\nDone.")